# Classify Job Postings

Run `3-scrape-job-postings.ipynb` first so that `jobs/1-scraped_jobs.jsonl` exists.

This notebook reads the scraped jobs, uses an LLM to classify whether each role is truly an AI engineering role, and writes **all** classified jobs (including non-AI engineering ones) to `jobs/2-classified-jobs.jsonl`. Subsequent notebooks filter by `is_ai_engineering_role` to work with only the accepted roles.

In [1]:
import json
from dotenv import load_dotenv
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import HTML, display

In [8]:

load_dotenv(override=True)

True

### Load scraped jobs from file

In [2]:
scraped_jobs_path = Path("jobs") / "1-scraped_jobs.jsonl"

if not scraped_jobs_path.exists():
    raise FileNotFoundError(
        f"Scraped jobs JSONL file not found: {scraped_jobs_path.resolve()}. Run 3-analyse-aie-job-postings.ipynb first."
    )

if scraped_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The scraped jobs JSONL file is empty. Run 3-analyse-aie-job-postings.ipynb first."
    )

jobs_df = pd.read_json(scraped_jobs_path, lines=True)

print(f"Scraped jobs JSONL file loaded successfully with {len(jobs_df)} entries!")

Scraped jobs JSONL file loaded successfully with 56 entries!


### Define the prompt

We define the instructions that tell the model what counts as an AI engineering role.

In [9]:
client = OpenAI()

instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models or in other words integrating them into products.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- MLOps and platform engineering are not AI engineering, as they focus on infrastructure and tooling rather than building AI-powered features.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
""".strip()

### Define the schema

We tell the model to return structured output with a boolean decision and a short reason.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

### Make the LLM calls

We now ask the model to classify each job posting.

In [ ]:
results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    description = job["description"]

    print(f"Screening job {i}/{len(jobs_df)}: {title}")

    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=instructions,
        input=f"""Classify this job posting.\n\nTitle: {title}\n\nDescription:\n{description}""",
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)
    is_ai_engineering_role = result["is_ai_engineering_role"]
    reason = result["reason"].strip()

    results.append(
        {
            "is_ai_engineering_role": is_ai_engineering_role,
            "llm_reason": reason,
        }
    )

### Combine and save the results

We join the model output back to the scraped jobs and save the accepted jobs

In [ ]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

results_df = pd.DataFrame(results)
screened_jobs = pd.concat([jobs_df, results_df], axis=1)

non_ai_engineer_count = int((~screened_jobs["is_ai_engineering_role"]).sum())
screened_jobs.to_json(
    classified_jobs_path, orient="records", lines=True, force_ascii=False
)

### Display the results

In [ ]:
ai_engineer_count = int(screened_jobs["is_ai_engineering_role"].sum())

print(f"Saved classified jobs to: {classified_jobs_path.resolve()}")
print(f"Jobs screened by LLM: {len(screened_jobs)}")
print(f"Jobs classified as AI engineering roles: {ai_engineer_count}")
print(f"Jobs classified as not AI engineering or unclear: {non_ai_engineer_count}")

results_to_show = screened_jobs[
    ["title", "is_ai_engineering_role", "llm_reason", "job_url"]
].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))